In [ ]:
import argparse
import math
import os
import time
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torchvision.models import resnet18

In [ ]:
# --- PerforatedAI imports -----------------------------------
# GPA (globals_perforatedai) holds the global configuration
# object (GPA.pc) and the training tracker (GPA.pai_tracker).
# UPA (utils_perforatedai) provides the initialize_pai()
# helper that wraps every eligible layer with PAI dendrite
# support.
from perforatedai import globals_perforatedai as GPA
from perforatedai import utils_perforatedai as UPA
# ------------------------------------------------------------

try:
    import wandb
except ImportError:
    wandb = None

try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    roc_auc_score = None

try:
    from fvcore.nn import FlopCountAnalysis
except ImportError:
    FlopCountAnalysis = None


_AUC_WARNING_EMITTED = False

In [ ]:
# Model definition
def resnet18_mnist() -> nn.Module:
    """
    Standard torchvision ResNet-18 adapted for grayscale 28×28 MNIST.

    Changes vs. ImageNet ResNet-18:
      - conv1: 1 input channel, 3×3 kernel, stride 1, no bias
        (the original 7×7 stride-2 kernel is too aggressive for 28×28)
      - maxpool replaced with Identity so spatial resolution is preserved
        through the first stem block
    """
    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(
        in_channels=1,
        out_channels=model.conv1.out_channels,
        kernel_size=3,
        stride=1,
        padding=1,
        bias=False,
    )
    model.maxpool = nn.Identity()
    return model

In [ ]:
args = argparse.Namespace(
    # W&B settings
    use_wandb=True,
    wandb_project="resnet-18_perf",
    wandb_entity="labebe",
    wandb_api_key="",        # intentionally blank - loaded from environment variable
    wandb_mode="online",
    wandb_anonymous="never",
    wandb_run_name="perf_ai_run2",
    wandb_run_id=None,
    wandb_resume=None,
)

In [ ]:
# W&B helpers  (unchanged from baseline)
def init_wandb(args) -> Optional["wandb.sdk.wandb_run.Run"]:
    if not args.use_wandb:
        print("W&B disabled. Pass --use-wandb to enable experiment logging.")
        return None

    if wandb is None:
        print("W&B logging requested but wandb is not installed.")
        return None

    if args.wandb_mode == "disabled":
        return None

    api_key = os.environ.get("WANDB_API_KEY", "") or args.wandb_api_key

    if args.wandb_mode == "offline":
        print("W&B running in offline mode.")

    if args.wandb_mode == "online" and not api_key and args.wandb_anonymous == "never":
        print("No WANDB_API_KEY found. Will try existing local wandb login credentials.")

    try:
        if api_key:
            wandb.login(key=api_key)
    except Exception as exc:
        print(f"W&B login failed: {exc}")


    run_config = vars(args).copy()
    run_config.pop("wandb_api_key", None)

    entity = args.wandb_entity if args.wandb_entity else None
    init_kwargs = dict(
        project=args.wandb_project,
        entity=entity,
        name=args.wandb_run_name if args.wandb_run_name else None,
        mode=args.wandb_mode,
        config=run_config,
        anonymous=args.wandb_anonymous,
        id=args.wandb_run_id if args.wandb_run_id else None,
        resume=args.wandb_resume if args.wandb_run_id else None,
    )

    try:
        run = wandb.init(**init_kwargs)
    except Exception as exc:
        print(f"W&B init failed: {exc}")
        if entity is not None:
            init_kwargs["entity"] = None
            try:
                run = wandb.init(**init_kwargs)
            except Exception as exc2:
                print(f"W&B init failed again: {exc2}")
                return None
        else:
            return None

    print(f"W&B initialized: project={args.wandb_project}, mode={args.wandb_mode}")
    return run

def log_to_wandb(run, metrics: Dict, step: Optional[int] = None) -> None:
    if run is None:
        return
    try:
        run.log(metrics, step=step)
    except Exception as exc:
        print(f"W&B log failed: {exc}")


def finish_wandb(run) -> None:
    if run is None:
        return
    try:
        run.finish()
    except Exception as exc:
        print(f"W&B finish failed: {exc}")


In [ ]:
init_wandb(args)

In [ ]:
# AUC helper  (unchanged from baseline)
# ============================================================
def compute_multiclass_auc(targets: torch.Tensor, probabilities: torch.Tensor) -> float:
    global _AUC_WARNING_EMITTED

    if roc_auc_score is None:
        if not _AUC_WARNING_EMITTED:
            print("AUC unavailable: scikit-learn is not installed.")
            _AUC_WARNING_EMITTED = True
        return float("nan")

    targets_np = targets.cpu().numpy()
    probs_np = probabilities.cpu().numpy()
    num_classes = probabilities.size(1)
    class_aucs = []
    present_classes = sorted(set(int(v) for v in targets_np.tolist()))

    try:
        for class_index in range(num_classes):
            binary_targets = (targets_np == class_index).astype(int)
            if binary_targets.sum() == 0 or binary_targets.shape[0] - binary_targets.sum() == 0:
                continue
            class_auc = roc_auc_score(binary_targets, probs_np[:, class_index])
            if not math.isnan(float(class_auc)):
                class_aucs.append(float(class_auc))

        if class_aucs:
            return float(sum(class_aucs) / len(class_aucs))

        if not _AUC_WARNING_EMITTED:
            print(f"AUC unavailable: no valid class AUCs. present_classes={present_classes}")
            _AUC_WARNING_EMITTED = True
        return float("nan")
    except Exception as exc:
        if not _AUC_WARNING_EMITTED:
            print(f"AUC unavailable: {exc}. present_classes={present_classes}")
            _AUC_WARNING_EMITTED = True
        return float("nan")

In [ ]:
# ============================================================
# Benchmarking helpers  (unchanged from baseline)
# ============================================================
def benchmark_inference_throughput(
    model: nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    max_batches: int = 20,
) -> float:
    model_was_training = model.training
    model.to(device)
    model.eval()
    total_inputs = 0

    with torch.no_grad():
        for idx, (data, _) in enumerate(data_loader):
            if idx >= 2:
                break
            _ = model(data.to(device))

        if device.type == "cuda":
            torch.cuda.synchronize(device)
        start = time.perf_counter()

        for idx, (data, _) in enumerate(data_loader):
            if idx >= max_batches:
                break
            _ = model(data.to(device))
            total_inputs += data.size(0)

        if device.type == "cuda":
            torch.cuda.synchronize(device)
        elapsed = time.perf_counter() - start

    if model_was_training:
        model.train()
    return float("nan") if elapsed <= 0 else float(total_inputs / elapsed)




def benchmark_latency_ms(
    model: nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device,
    max_batches: int = 20,
) -> float:
    model_was_training = model.training
    model.to(device)
    model.eval()
    total_batches = 0

    with torch.no_grad():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        start = time.perf_counter()

        for idx, (data, _) in enumerate(data_loader):
            if idx >= max_batches:
                break
            _ = model(data.to(device))
            total_batches += 1

        if device.type == "cuda":
            torch.cuda.synchronize(device)
        elapsed = time.perf_counter() - start

    if model_was_training:
        model.train()
    return float("nan") if total_batches == 0 else float((elapsed / total_batches) * 1000.0)

    def benchmark_cpu_latency_single_core_ms(
    model: nn.Module,
    data_loader: torch.utils.data.DataLoader,
    max_batches: int = 20,
) -> float:
    previous_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(1)
        return benchmark_latency_ms(model, data_loader, torch.device("cpu"), max_batches=max_batches)
    finally:
        torch.set_num_threads(previous_threads)


def estimate_flops_with_hooks(
    model: nn.Module,
    device: torch.device,
    input_shape: Tuple[int, int, int, int] = (1, 1, 28, 28),
) -> float:
    total_flops = 0.0
    handles = []
    model_was_training = model.training

    def conv_hook(module: nn.Conv2d, inputs, output):
        nonlocal total_flops
        b, oc, oh, ow = output.shape
        kh, kw = module.kernel_size
        mul = (module.in_channels // module.groups) * kh * kw
        bias_ops = 1 if module.bias is not None else 0
        total_flops += b * oc * oh * ow * ((2 * mul) + bias_ops)

    def linear_hook(module: nn.Linear, inputs, output):
        nonlocal total_flops
        batch_size = 1 if output.dim() == 1 else output.shape[0]
        out_features = output.shape[0] if output.dim() == 1 else output.shape[-1]
        bias_ops = 1 if module.bias is not None else 0
        total_flops += batch_size * out_features * ((2 * module.in_features) + bias_ops)

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            handles.append(module.register_forward_hook(conv_hook))
        elif isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(linear_hook))

    sample_input = torch.randn(*input_shape, device=device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        _ = model(sample_input)

    for h in handles:
        h.remove()

    if model_was_training:
        model.train()
    return float(total_flops)

def compute_model_stats(model: nn.Module, device: torch.device) -> Tuple[int, float, str]:
    model.to(device)
    param_count = sum(p.numel() for p in model.parameters())
    flops = float("nan")
    flops_source = "unavailable"

    if FlopCountAnalysis is not None:
        try:
            sample_input = torch.randn(1, 1, 28, 28, device=device)
            flops = float(FlopCountAnalysis(model, sample_input).total())
            flops_source = "fvcore"
        except Exception:
            flops = float("nan")

    if math.isnan(flops):
        try:
            flops = estimate_flops_with_hooks(model, device)
            flops_source = "approximate_hooks"
        except Exception as exc:
            print(f"FLOPS fallback failed: {exc}")

    return param_count, flops, flops_source
def safe_number(value) -> Optional[float]:
    if isinstance(value, torch.Tensor):
        value = value.item()
    value = float(value)
    return None if (math.isnan(value) or math.isinf(value)) else value


def update_min_max(stats: Dict[str, float], key: str, value) -> None:
    value = safe_number(value)
    if value is None:
        return
    stats[f"{key}_min"] = min(stats.get(f"{key}_min", value), value)
    stats[f"{key}_max"] = max(stats.get(f"{key}_max", value), value)



In [ ]:
# Training loop
# ============================================================
def train(
    args,
    model: nn.Module,
    device: torch.device,
    train_loader: torch.utils.data.DataLoader,
    optimizer: optim.Optimizer,
    epoch: int,
) -> float:
    """
    One full pass over the training set.

    PAI change vs. baseline:
      After the batch loop finishes, we call
      GPA.pai_tracker.add_extra_score() to hand the training
      accuracy to the PAI tracker.  PAI uses this alongside the
      validation score (reported in test()) to judge whether the
      current dendrite configuration is learning effectively.
    """
    model.train()
    model.to(device)
    correct = 0


    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)            # logits (ResNet-18 has no softmax)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % args.log_interval == 0:
            print(
                "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                    epoch,
                    batch_idx * len(data),
                    len(train_loader.dataset),
                    100.0 * batch_idx / len(train_loader),
                    loss.item(),
                )
            )
            if args.dry_run:
                break

        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum()

    train_accuracy = 100.0 * correct.item() / len(train_loader.dataset)


    # --- PAI: report training accuracy to the tracker ----------
    # The tracker accumulates both training and validation scores
    # to decide when validation improvement has stalled, which is
    # the trigger for adding the next dendrite.
    GPA.pai_tracker.add_extra_score(train_accuracy, "train")
    # PAI may have moved the model internally; re-attach to device.
    model.to(device)
    # -----------------------------------------------------------

    return train_accuracy




In [ ]:
# ============================================================
# Evaluation helper  (unchanged from baseline)
# ============================================================
def evaluate(
    model: nn.Module,
    device: torch.device,
    data_loader: torch.utils.data.DataLoader,
) -> Dict[str, float]:
    """Run the model in eval mode and compute loss, accuracy, AUC, precision@1."""
    model.eval()
    model.to(device)
    avg_loss = 0.0
    correct = 0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)                        # logits
            probs = torch.softmax(output, dim=1)        # probabilities for AUC

            avg_loss += F.cross_entropy(output, target, reduction="sum").item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum()

            all_targets.append(target.detach().cpu())
            all_probs.append(probs.detach().cpu())

    avg_loss /= len(data_loader.dataset)
    accuracy = 100.0 * correct.item() / len(data_loader.dataset)
    all_targets = torch.cat(all_targets)
    all_probs = torch.cat(all_probs)
    auc = compute_multiclass_auc(all_targets, all_probs)
    precision_at_1 = accuracy  # Precision@1 == top-1 accuracy for single-label tasks

    return {"loss": avg_loss, "accuracy": accuracy, "auc": auc, "precision_at_1": precision_at_1}



In [ ]:
# ============================================================
# Validation + test loop (PAI-aware)
# ============================================================
def test(
    model: nn.Module,
    device: torch.device,
    val_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    optimizer: optim.Optimizer,
    scheduler,
    args,
) -> Tuple[nn.Module, optim.Optimizer, object, bool, Dict[str, float]]:
    """
    Evaluate on validation and test sets, then let PAI decide
    whether to add a dendrite.

    PAI changes vs. baseline:
      1. After evaluating, we call
         GPA.pai_tracker.add_validation_score(accuracy, model).
         PAI returns:
           - model         : possibly restructured (new dendrite added)
           - restructured  : True if a dendrite was just added
           - training_complete : True if PAI has finished (max dendrites
                                  reached AND no further improvement is
                                  expected, or another stop condition met)

      2. If restructured is True, the optimizer and scheduler must be
         re-created via PAI's setup_optimizer() so they track the new
         parameters that were added.

      3. training_complete is surfaced to main() so the epoch loop can
         exit early when PAI decides training is done.
    """
    val_metrics = evaluate(model, device, val_loader)
    test_metrics = evaluate(model, device, test_loader)


    print(
        "\nValidation set: Average loss: {:.4f}, Accuracy: {:.2f}%, AUC: {}\n".format(
            val_metrics["loss"],
            val_metrics["accuracy"],
            "nan" if safe_number(val_metrics["auc"]) is None else "{:.6f}".format(val_metrics["auc"]),
        )
    )
    print(
        "Test set: Average loss: {:.4f}, Accuracy: {:.2f}%, AUC: {}\n".format(
            test_metrics["loss"],
            test_metrics["accuracy"],
            "nan" if safe_number(test_metrics["auc"]) is None else "{:.6f}".format(test_metrics["auc"]),
        )
    )

    # --- PAI: hand the validation accuracy to the tracker ------
    # The tracker monitors whether accuracy is still improving.
    # When improvement stalls (over n_epochs_to_switch epochs by
    # default), it loads the best checkpoint it has seen so far
    # and adds one dendrite — up to max_dendrites (set to 3).
    # It returns the (possibly modified) model and status flags.
    model, restructured, training_complete = GPA.pai_tracker.add_validation_score(
        val_metrics["accuracy"], model
    )
    # PAI may have moved the model; ensure it is on the right device.
    model.to(device)

    # If a dendrite was just added the model has new parameters, so
    # the existing optimizer no longer covers them.  Re-create both
    # optimizer and scheduler through PAI so they are registered
    # correctly for any future restructuring too.
    if restructured:
        print("PAI restructured the model — resetting optimizer and scheduler.")
        optimArgs = {"params": model.parameters(), "lr": args.lr}
        schedArgs = {"step_size": 1, "gamma": args.gamma}
        optimizer, scheduler = GPA.pai_tracker.setup_optimizer(model, optimArgs, schedArgs)
    # -----------------------------------------------------------


    metrics = {
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"],
        "validation_auc": val_metrics["auc"],
        "validation_precision_at_1": val_metrics["precision_at_1"],
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_auc": test_metrics["auc"],
        "test_precision_at_1": test_metrics["precision_at_1"],
    }
    return model, optimizer, scheduler, training_complete, metrics



In [ ]:

# Entry point
# ============================================================
def main() -> None:
    parser = argparse.ArgumentParser(
        description="PyTorch MNIST ResNet-18 with PerforatedAI (3 dendrites)."
    )
    parser.add_argument("--batch-size", type=int, default=64, metavar="N")
    parser.add_argument("--test-batch-size", type=int, default=1000, metavar="N")
    parser.add_argument("--val-split", type=float, default=0.1, metavar="N")
    parser.add_argument("--epochs", type=int, default=50, metavar="N")
    parser.add_argument("--lr", type=float, default=1.0, metavar="LR")
    parser.add_argument("--gamma", type=float, default=0.7, metavar="M")
    parser.add_argument("--no-cuda", action="store_true", default=False)
    parser.add_argument("--no-mps", action="store_true", default=False)
    parser.add_argument("--dry-run", action="store_true", default=False)
    parser.add_argument("--seed", type=int, default=1, metavar="S")
    parser.add_argument("--log-interval", type=int, default=10, metavar="N")
    parser.add_argument("--save-model", action="store_true", default=False)
    parser.add_argument("--checkpoint-path", type=str, default="resnet18_pai_last.pt")
    parser.add_argument("--best-model-path", type=str, default="resnet18_pai_best.pt")
    parser.add_argument("--resume-from-checkpoint", action="store_true", default=False)
    parser.add_argument("--output-dir", type=str, default="outputs")
    parser.add_argument(
        "--data-root",
        type=str,
        default="/ocean/projects/cis260045p/shared/data",
        help="Root directory for MNIST data (PSC default or local path).",
    )

    parser.add_argument("--use-wandb", action="store_true", default=False)
    parser.add_argument("--wandb-project", type=str, default="MNIST_PERF")
    parser.add_argument("--wandb-entity", type=str, default="")
    parser.add_argument("--wandb-run-name", type=str, default="")
    parser.add_argument("--wandb-mode", type=str, default="online",
                        choices=["online", "offline", "disabled"])
    parser.add_argument("--wandb-api-key", type=str, default="")
    parser.add_argument("--wandb-run-id", type=str, default="")
    parser.add_argument("--wandb-resume", type=str, default="allow",
                        choices=["allow", "must", "never"])
    parser.add_argument("--wandb-anonymous", type=str, default="never",
                        choices=["never", "allow", "must"])

    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    checkpoint_path = os.path.join(args.output_dir, args.checkpoint_path)
    best_model_path = os.path.join(args.output_dir, args.best_model_path)

    use_cuda = (not args.no_cuda) and torch.cuda.is_available()
    use_mps = (not args.no_mps) and torch.backends.mps.is_available()
    torch.manual_seed(args.seed)

    if use_cuda:
        device = torch.device("cuda")
    elif use_mps:
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])


    data_root = args.data_root
    try:
        train_full = datasets.MNIST(data_root, train=True, download=False, transform=transform)
        test_dataset = datasets.MNIST(data_root, train=False, download=False, transform=transform)
    except RuntimeError:
        train_full = datasets.MNIST(data_root, train=True, download=True, transform=transform)
        test_dataset = datasets.MNIST(data_root, train=False, download=True, transform=transform)

    total_train = len(train_full)
    val_size = max(1, int(total_train * args.val_split))
    train_size = total_train - val_size
    train_dataset, val_dataset = torch.utils.data.random_split(
        train_full,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(args.seed),
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=True,
        num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=args.test_batch_size, shuffle=False,
        num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=args.test_batch_size, shuffle=False,
        num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
    )

    cpu_benchmark_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
    gpu_benchmark_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False)


In [ ]:
 GPA.pc.set_unwrapped_modules_confirmed(True)

In [ ]:
GPA.pc.set_max_dendrites(3)
GPA.pc.set_testing_dendrite_capacity(False)
GPA.pc.set_unwrapped_modules_confirmed(True)  # <-- add this

model = resnet18_mnist()
model = UPA.initialize_pai(model)
model = model.to(device)

In [ ]:
import argparse, os, torch

args = argparse.Namespace(
    lr=1.0, gamma=0.7, resume_from_checkpoint=False,
    batch_size=64, test_batch_size=1000, val_split=0.1,
    epochs=50, seed=1, log_interval=10, save_model=False,
    checkpoint_path="resnet18_pai_last.pt",
    best_model_path="resnet18_pai_best.pt",
    output_dir="outputs",
    data_root="/ocean/projects/cis260045p/shared/data",
    use_wandb=False, wandb_project="MNIST_PERF",
    wandb_entity="", wandb_run_name="", wandb_mode="disabled",
    wandb_api_key="", wandb_run_id="", wandb_resume="allow",
    wandb_anonymous="never", no_cuda=False, no_mps=False, dry_run=False
)
os.makedirs(args.output_dir, exist_ok=True)
checkpoint_path = os.path.join(args.output_dir, args.checkpoint_path)
best_model_path = os.path.join(args.output_dir, args.best_model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# PAI global configuration
    # =========================================================
    # set_max_dendrites(3)
    #   This is the key setting.  PerforatedAI will try to add
    #   dendrites one at a time whenever validation accuracy
    #   plateaus, but it will stop once 8 dendrites have been
    #   added to the network in total.
    GPA.pc.set_max_dendrites(8)

    # set_testing_dendrite_capacity(False)
    #   Disables an internal PAI routine that probes how much
    #   capacity each dendrite has.  Keeping it False is the
    #   standard setting for training runs (same as mnist_perf.py).
    GPA.pc.set_testing_dendrite_capacity(False)
    # =========================================================

    # Build the ResNet-18 MNIST model and hand it to PAI.
    # UPA.initialize_pai() inspects every nn.Linear and nn.Conv2d
    # layer and wraps them with PAI's PB (perforated branch) nodes
    # so that dendrites can be attached at training time.
    model = resnet18_mnist()
    model = UPA.initialize_pai(model)
    model = model.to(device)

    # =========================================================
    # Optimizer / scheduler — set up through the PAI tracker
    # =========================================================
    # We register the optimizer type and scheduler type with PAI
    # first.  When PAI adds a dendrite and restructures the model
    # it needs to know how to recreate them with the updated
    # parameter list; registering them here makes that automatic.
    GPA.pai_tracker.set_optimizer(optim.Adadelta)
    GPA.pai_tracker.set_scheduler(StepLR)
    optimArgs = {"params": model.parameters(), "lr": args.lr}
    schedArgs = {"step_size": 1, "gamma": args.gamma}
    optimizer, scheduler = GPA.pai_tracker.setup_optimizer(model, optimArgs, schedArgs)
    # =========================================================


    run = init_wandb(args)
    cycle_start = time.perf_counter()
    start_epoch = 1
    running_stats: Dict[str, float] = {}
    best_validation_accuracy = float("-inf")
    best_validation_snapshot: Dict[str, float] = {}

    if args.resume_from_checkpoint and os.path.exists(checkpoint_path):
        try:
            ckpt = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            running_stats = ckpt.get("running_stats", {})
            best_validation_accuracy = ckpt.get("best_validation_accuracy", float("-inf"))
            best_validation_snapshot = ckpt.get("best_validation_snapshot", {})
            start_epoch = int(ckpt.get("epoch", 0)) + 1
            cycle_start = time.perf_counter() - float(ckpt.get("seconds_per_training_cycle", 0.0))
            print(f"Resuming from epoch {start_epoch}.")
        except Exception as exc:
            print(f"Failed to resume checkpoint: {exc}")





In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

data_root = args.data_root
try:
    train_full = datasets.MNIST(data_root, train=True, download=False, transform=transform)
    test_dataset = datasets.MNIST(data_root, train=False, download=False, transform=transform)
except RuntimeError:
    train_full = datasets.MNIST(data_root, train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(data_root, train=False, download=True, transform=transform)

total_train = len(train_full)
val_size = max(1, int(total_train * args.val_split))
train_size = total_train - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    train_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(args.seed),
)

use_cuda = torch.cuda.is_available() and not args.no_cuda
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=args.batch_size, shuffle=True,
    num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=args.test_batch_size, shuffle=False,
    num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=args.test_batch_size, shuffle=False,
    num_workers=1 if use_cuda else 0, pin_memory=use_cuda,
)
cpu_benchmark_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)
gpu_benchmark_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False)
print("Data loaders ready.")


In [ ]:
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

In [ ]:
# Main training loop
    # =========================================================
    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.perf_counter()

        train_accuracy = train(args, model, device, train_loader, optimizer, epoch)

        # test() now returns the (possibly restructured) model and
        # the training_complete flag from PAI.
        model, optimizer, scheduler, training_complete, eval_metrics = test(
            model, device, val_loader, test_loader, optimizer, scheduler, args
        )

        if scheduler is not None:
            scheduler.step()

        seconds_per_training_epoch = time.perf_counter() - epoch_start
        seconds_per_training_cycle = time.perf_counter() - cycle_start

        validation_accuracy = eval_metrics["validation_accuracy"]
        validation_auc = eval_metrics["validation_auc"]

        update_min_max(running_stats, "validation_accuracy", validation_accuracy)
        update_min_max(running_stats, "validation_auc", validation_auc)
        update_min_max(running_stats, "test_accuracy", eval_metrics["test_accuracy"])
        update_min_max(running_stats, "test_auc", eval_metrics["test_auc"])

        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_validation_snapshot = {
                "test_accuracy_at_best_validation": eval_metrics["test_accuracy"],
                "test_auc_at_best_validation": eval_metrics["test_auc"],
                "validation_accuracy_best": validation_accuracy,
                "validation_auc_at_best_validation": validation_auc,
                "epoch_at_best_validation": epoch,
            }
            try:
                torch.save(model.state_dict(), best_model_path)
            except Exception as exc:
                print(f"Failed to save best model: {exc}")

        epoch_log: Dict[str, Optional[float]] = {
            "epoch": epoch,
            "perforatedai/train_accuracy": train_accuracy,
            "perforatedai/validation_accuracy": validation_accuracy,
            "perforatedai/validation_accuracy_min": running_stats.get("validation_accuracy_min"),
            "perforatedai/validation_accuracy_max": running_stats.get("validation_accuracy_max"),
            "perforatedai/validation_auc": safe_number(validation_auc),
            "perforatedai/validation_auc_min": running_stats.get("validation_auc_min"),
            "perforatedai/validation_auc_max": running_stats.get("validation_auc_max"),
            "perforatedai/test_accuracy": eval_metrics["test_accuracy"],
            "perforatedai/test_accuracy_min": running_stats.get("test_accuracy_min"),
            "perforatedai/test_accuracy_max": running_stats.get("test_accuracy_max"),
            "perforatedai/test_auc": safe_number(eval_metrics["test_auc"]),
            "perforatedai/test_auc_min": running_stats.get("test_auc_min"),
            "perforatedai/test_auc_max": running_stats.get("test_auc_max"),
            "perforatedai/precision_at_1": eval_metrics["test_precision_at_1"],
            "perforatedai/seconds_per_training_epoch": seconds_per_training_epoch,
            "perforatedai/seconds_per_training_cycle": seconds_per_training_cycle,
        }

        if best_validation_snapshot:
            epoch_log["perforatedai/test_accuracy_at_best_validation"] = (
                best_validation_snapshot["test_accuracy_at_best_validation"]
            )
            epoch_log["perforatedai/test_auc_at_best_validation"] = safe_number(
                best_validation_snapshot["test_auc_at_best_validation"]
            )
            epoch_log["perforatedai/epoch_at_best_validation"] = (
                best_validation_snapshot["epoch_at_best_validation"]
            )

        print(f"Epoch {epoch} metrics: {epoch_log}")
        if run is not None:
            log_to_wandb(run, epoch_log, step=epoch)

        # Save a last-epoch checkpoint every epoch so training can
        # be resumed if interrupted.
        ckpt_data = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "running_stats": running_stats,
            "best_validation_accuracy": best_validation_accuracy,
            "best_validation_snapshot": best_validation_snapshot,
            "seconds_per_training_cycle": seconds_per_training_cycle,
        }
        try:
            torch.save(ckpt_data, checkpoint_path)
        except Exception as exc:
            print(f"Failed to save checkpoint: {exc}")

        # PAI sets training_complete=True when it has exhausted all
        # max_dendrites (3 here) and further improvement is not
        # expected, signalling that training can stop early.
        if training_complete:
            print("PAI signalled training complete — stopping early.")
            break
    # =========================================================

    model.eval()

    gpu_inference_ips = float("nan")
    if torch.cuda.is_available() and not args.no_cuda:
        gpu_inference_ips = benchmark_inference_throughput(
            model, gpu_benchmark_loader, torch.device("cuda")
        )

    model_cpu = model.to(torch.device("cpu"))
    cpu_inference_ips = benchmark_inference_throughput(model_cpu, cpu_benchmark_loader, torch.device("cpu"))
    latency_ms = benchmark_cpu_latency_single_core_ms(model_cpu, cpu_benchmark_loader)
    param_count, flops, flops_source = compute_model_stats(model_cpu, torch.device("cpu"))

    final_metrics: Dict[str, object] = {
        "perforatedai/gpu_inference_inputs_per_second": safe_number(gpu_inference_ips),
        "perforatedai/cpu_inference_inputs_per_second": safe_number(cpu_inference_ips),
        "efficientnet/num_parameters": param_count,
        "efficientnet/flops": safe_number(flops),
        "efficientnet/flops_source": flops_source,
        "efficientnet/latency_ms_per_batch": safe_number(latency_ms),
    }

    if best_validation_snapshot and safe_number(latency_ms) is not None:
        final_metrics["efficientnet/accuracy_vs_flops"] = (
            best_validation_snapshot["test_accuracy_at_best_validation"]
        )
        final_metrics["perforatedai/test_auc_at_best_validation"] = safe_number(
            best_validation_snapshot["test_auc_at_best_validation"]
        )
        final_metrics["perforatedai/validation_accuracy_best"] = (
            best_validation_snapshot["validation_accuracy_best"]
        )
        final_metrics["perforatedai/validation_auc_at_best_validation"] = safe_number(
            best_validation_snapshot["validation_auc_at_best_validation"]
        )
        final_metrics["perforatedai/epoch_at_best_validation"] = (
            best_validation_snapshot["epoch_at_best_validation"]
        )

    print(f"Final performance metrics: {final_metrics}")
    if run is not None:
        log_to_wandb(run, final_metrics)
        finish_wandb(run)

    if args.save_model:
        torch.save(model.state_dict(), "resnet18_perforatedai_mnist.pt")

In [ ]:
#run this last bc auc was nan for the first run ... result (Test AUC: 0.4910)

In [ ]:
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

model.eval()
all_labels = []
all_probs = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        probs = F.softmax(output, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(target.cpu().numpy())

from sklearn.preprocessing import label_binarize
import numpy as np
all_labels_bin = label_binarize(all_labels, classes=list(range(10)))
auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr')
print(f"Test AUC: {auc:.4f}")